# 🎯 Data Augmentation

## Phase 2: Multi-Source TTS Data Generation

### Objective
Our current dataset is **mono-algorithmic**—all fake samples come from a single VITS model. If we train now, the model will learn to detect **VITS specifically**, not **Deepfakes generally**.

This notebook generates diverse synthetic Bengali audio using:
1. **Crikk API** (~1,500 samples) - Proprietary TTS model
2. **Orpheus Bangla TTS** (~1,500 samples) - Transformer-based model

### Why This Matters
- VITS uses CNN/Flow architecture
- Crikk likely uses WaveNet/Tacotron-based architecture  
- Orpheus uses Transformer architecture (similar to LLaMA)

Training on all three forces the detector to learn **generalized features of synthesis** rather than architecture-specific glitches.

---


## 📁 1. Setup Directory Structure & Dependencies


In [1]:
import os
import sys
import shutil
from pathlib import Path

# Define paths
PROJECT_ROOT = Path(r"F:/ML Project")
DATA_ANALYSIS_DIR = PROJECT_ROOT / "Bangla Audio Deepfake Detection" / "Data Analysis"
AUGMENTATION_DIR = PROJECT_ROOT / "Bangla Audio Deepfake Detection" / "Data Augmentation"
FINAL_DATA_DIR = PROJECT_ROOT / "final_data"

# Create augmentation subdirectories
AUGMENTATION_DATA_DIR = AUGMENTATION_DIR / "Data"
AUGMENTATION_MANIFEST_DIR = AUGMENTATION_DIR / "Manifest"

# Create directories for each TTS source
ORPHEUS_OUTPUT_DIR = AUGMENTATION_DATA_DIR / "orpheus_deepfake"

# Create all directories
for dir_path in [AUGMENTATION_DATA_DIR, AUGMENTATION_MANIFEST_DIR, ORPHEUS_OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {dir_path}")

print("\n📂 Directory structure created successfully!")


✅ Created: F:\ML Project\Bangla Audio Deepfake Detection\Data Augmentation\Data
✅ Created: F:\ML Project\Bangla Audio Deepfake Detection\Data Augmentation\Manifest
✅ Created: F:\ML Project\Bangla Audio Deepfake Detection\Data Augmentation\Data\orpheus_deepfake

📂 Directory structure created successfully!


In [ ]:
# Install required packages
# %pip install pandas numpy tqdm requests soundfile scipy transformers torch torchaudio accelerate snac


  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/110.9 MB 1.3 MB/s eta 0:01:23
   ---------------------------------------- 0.5/110.9 MB 1.3 MB/s eta 0:01:23
   -------------------------


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import requests
import soundfile as sf
import time
import json
import random
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

print("📦 All imports successful!")


📦 All imports successful!


## 📋 2. Copy Original Manifests


In [3]:
# Copy original manifests to augmentation directory
SOURCE_MANIFEST_DIR = DATA_ANALYSIS_DIR / "Manifest"
manifest_files = ["train_manifest.csv", "test_manifest.csv", "val_manifest.csv", "dataset_manifest.csv"]

for manifest_file in manifest_files:
    src = SOURCE_MANIFEST_DIR / manifest_file
    dst = AUGMENTATION_MANIFEST_DIR / manifest_file
    if src.exists():
        shutil.copy(src, dst)
        print(f"✅ Copied: {manifest_file}")
    else:
        print(f"⚠️ Not found: {manifest_file}")

print("\n📋 Manifests copied successfully!")


✅ Copied: train_manifest.csv
✅ Copied: test_manifest.csv
✅ Copied: val_manifest.csv
✅ Copied: dataset_manifest.csv

📋 Manifests copied successfully!


In [4]:
# Load and display current manifest statistics
train_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "train_manifest.csv")
test_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "test_manifest.csv")
val_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "val_manifest.csv")

print("📊 Current Dataset Statistics:")
print("="*50)
print(f"Training set: {len(train_df)} samples")
print(f"  - Real: {len(train_df[train_df['label'] == 'real'])}")
print(f"  - Fake: {len(train_df[train_df['label'] == 'fake'])}")
print(f"\nValidation set: {len(val_df)} samples")
print(f"  - Real: {len(val_df[val_df['label'] == 'real'])}")
print(f"  - Fake: {len(val_df[val_df['label'] == 'fake'])}")
print(f"\nTest set: {len(test_df)} samples")
print(f"  - Real: {len(test_df[test_df['label'] == 'real'])}")
print(f"  - Fake: {len(test_df[test_df['label'] == 'fake'])}")

print("\n📈 Source Distribution in Training Set:")
print(train_df['source'].value_counts())


📊 Current Dataset Statistics:
Training set: 18614 samples
  - Real: 9657
  - Fake: 8957

Validation set: 3989 samples
  - Real: 2070
  - Fake: 1919

Test set: 3989 samples
  - Real: 2069
  - Fake: 1920

📈 Source Distribution in Training Set:
source
sust       13998
mozilla     3916
news         700
Name: count, dtype: int64


## 📝 3. Load Transcripts from Metadata Files


In [5]:
def load_mozilla_metadata() -> pd.DataFrame:
    """Load metadata from Mozilla Common Voice (pipe-separated)"""
    metadata_path = FINAL_DATA_DIR / "deepfake_data_mozilla" / "metadata.csv"
    df = pd.read_csv(metadata_path, sep='|')
    df.columns = ['filename', 'text']
    df['source'] = 'mozilla'
    return df

def load_news_metadata() -> pd.DataFrame:
    """Load metadata from News dataset (comma-separated)"""
    metadata_path = FINAL_DATA_DIR / "deepfake_data_news" / "metadata.csv"
    df = pd.read_csv(metadata_path)
    # Handle the format: filename, text (with leading space in text)
    df.columns = ['filename', 'text']
    df['text'] = df['text'].str.strip()
    df['source'] = 'news'
    return df

def load_sust_metadata() -> pd.DataFrame:
    """Load metadata from SUST TTS Corpus (pipe-separated)"""
    metadata_path = FINAL_DATA_DIR / "deepfake_data_sust" / "metadata.txt"
    df = pd.read_csv(metadata_path, sep='|', header=None, names=['filename', 'text'])
    df['source'] = 'sust'
    return df

# Load all metadata
mozilla_meta = load_mozilla_metadata()
news_meta = load_news_metadata()
sust_meta = load_sust_metadata()

print("📝 Transcript Statistics:")
print("="*50)
print(f"Mozilla Common Voice: {len(mozilla_meta)} transcripts")
print(f"News Dataset: {len(news_meta)} transcripts")
print(f"SUST TTS Corpus: {len(sust_meta)} transcripts")

# Combine all transcripts
all_transcripts = pd.concat([mozilla_meta, news_meta, sust_meta], ignore_index=True)
print(f"\n📊 Total transcripts available: {len(all_transcripts)}")


📝 Transcript Statistics:
Mozilla Common Voice: 2797 transcripts
News Dataset: 1000 transcripts
SUST TTS Corpus: 9999 transcripts

📊 Total transcripts available: 13796


In [6]:
# Display sample transcripts
print("📜 Sample Transcripts from Each Source:")
print("="*70)

for source in ['mozilla', 'news', 'sust']:
    sample = all_transcripts[all_transcripts['source'] == source].iloc[0]
    print(f"\n[{source.upper()}]")
    print(f"  Filename: {sample['filename']}")
    print(f"  Text: {sample['text'][:100]}..." if len(str(sample['text'])) > 100 else f"  Text: {sample['text']}")


📜 Sample Transcripts from Each Source:

[MOZILLA]
  Filename: common_voice_s5_0
  Text: ফলশ্রুতিতে নাটাল দলকে পরাজিত করে কারি কাপের শিরোপা জয়ে সবিশেষ ভূমিকা পালন করেন

[NEWS]
  Filename: 0
  Text: গতকাল মঙ্গলবার সকালে কলেজ মিলনায়তনে এ ঘটনা ঘটে

[SUST]
  Filename: 1001
  Text: উনিশ মে দুপুর তিনটায় হোটেল সুন্দরবনে আপনার একটি রিজার্ভেশন আছে


In [7]:
# Filter transcripts for optimal TTS synthesis
# Remove very short or very long transcripts
MIN_CHARS = 20
MAX_CHARS = 300

all_transcripts['text_length'] = all_transcripts['text'].astype(str).str.len()
filtered_transcripts = all_transcripts[
    (all_transcripts['text_length'] >= MIN_CHARS) & 
    (all_transcripts['text_length'] <= MAX_CHARS)
].copy()

print(f"📊 Filtered transcripts ({MIN_CHARS}-{MAX_CHARS} chars): {len(filtered_transcripts)}")
print(f"\nText length distribution:")
print(filtered_transcripts['text_length'].describe())


📊 Filtered transcripts (20-300 chars): 13183

Text length distribution:
count    13183.000000
mean        61.169916
std         28.969940
min         20.000000
25%         38.000000
50%         55.000000
75%         79.000000
max        170.000000
Name: text_length, dtype: float64


## 🎙️ 4. Crikk TTS Generation

Crikk is a proprietary TTS service with high-quality Bengali voices. We'll use their API to generate ~1,500 fake samples.

**Note:** You'll need to obtain API credentials from [Crikk](https://crikk.com/text-to-speech/bengali/)


In [ ]:
# Crikk API Configuration
# You'll need to sign up at https://crikk.com and get your API key

class CrikkTTS:
    """
    Crikk Text-to-Speech API wrapper for Bengali.
    
    Crikk offers multiple Bengali voices with different genders and styles.
    """
    
    def __init__(self, api_key: str = None):
        self.api_key = api_key or os.environ.get('CRIKK_API_KEY')
        self.base_url = "https://api.crikk.com/v1"
        
        # Bengali voices available in Crikk
        # Note: Check Crikk's documentation for the latest voice IDs
        self.voices = {
            'female_1': 'bn-BD-NabanitaNeural',  # Female voice
            'male_1': 'bn-BD-PradeepNeural',     # Male voice
            'female_2': 'bn-IN-TanishaaNeural',  # Indian Bengali Female
            'male_2': 'bn-IN-BashkarNeural',     # Indian Bengali Male
        }
        
    def synthesize(self, text: str, voice_id: str = 'female_1', 
                   output_path: str = None) -> Optional[bytes]:
        """
        Synthesize Bengali text to speech using Crikk API.
        
        Args:
            text: Bengali text to synthesize
            voice_id: Voice identifier (see self.voices)
            output_path: Optional path to save the audio file
            
        Returns:
            Audio bytes if successful, None otherwise
        """
        if not self.api_key:
            raise ValueError("Crikk API key not set. Set CRIKK_API_KEY environment variable.")
        
        voice_name = self.voices.get(voice_id, voice_id)
        
        headers = {
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json'
        }
        
        payload = {
            'text': text,
            'voice': voice_name,
            'format': 'wav',
            'sample_rate': 22050
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/tts",
                headers=headers,
                json=payload,
                timeout=60
            )
            
            if response.status_code == 200:
                audio_data = response.content
                if output_path:
                    with open(output_path, 'wb') as f:
                        f.write(audio_data)
                return audio_data
            else:
                print(f"Error: {response.status_code} - {response.text}")
                return None
                
        except Exception as e:
            print(f"Request failed: {e}")
            return None
    
    def batch_synthesize(self, texts: List[str], output_dir: Path,
                         voice_ids: List[str] = None,
                         start_idx: int = 0) -> List[Dict]:
        """
        Batch synthesize multiple texts.
        
        Args:
            texts: List of Bengali texts
            output_dir: Directory to save audio files
            voice_ids: List of voice IDs to use (cycles through if shorter than texts)
            start_idx: Starting index for filenames
            
        Returns:
            List of generation results with metadata
        """
        if voice_ids is None:
            voice_ids = list(self.voices.keys())
        
        results = []
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        for i, text in enumerate(tqdm(texts, desc="Generating Crikk samples")):
            voice_id = voice_ids[i % len(voice_ids)]
            filename = f"crikk_{start_idx + i:05d}.wav"
            output_path = output_dir / filename
            
            audio = self.synthesize(text, voice_id, str(output_path))
            
            results.append({
                'filename': filename,
                'text': text,
                'voice_id': voice_id,
                'success': audio is not None,
                'path': str(output_path) if audio else None
            })
            
            # Rate limiting
            time.sleep(0.5)
        
        return results

print("✅ CrikkTTS class defined")


In [ ]:
# ============================================
# CRIKK GENERATION - CONFIGURATION
# ============================================

# Configuration
CRIKK_API_KEY = None  # Set your API key here or use environment variable
CRIKK_TARGET_SAMPLES = 1500
CRIKK_TRAIN_RATIO = 0.8  # 80% train, 10% val, 10% test

# Sample transcripts for Crikk generation
# Using a mix of all sources for diversity
random.seed(42)
crikk_transcripts = filtered_transcripts.sample(n=min(CRIKK_TARGET_SAMPLES, len(filtered_transcripts)))
crikk_texts = crikk_transcripts['text'].tolist()

print(f"📊 Prepared {len(crikk_texts)} transcripts for Crikk synthesis")
print(f"\n🔑 API Key Status: {'✅ Set' if CRIKK_API_KEY or os.environ.get('CRIKK_API_KEY') else '❌ Not Set'}")

if not (CRIKK_API_KEY or os.environ.get('CRIKK_API_KEY')):
    print("\n⚠️ To generate Crikk samples:")
    print("1. Sign up at https://crikk.com")
    print("2. Get your API key")
    print("3. Set CRIKK_API_KEY variable above or as environment variable")
    print("\nAlternatively, you can use the manual web-based method below.")


In [ ]:
# Generate Crikk samples (uncomment and run when API key is set)

def generate_crikk_samples():
    """Generate samples using Crikk API"""
    crikk = CrikkTTS(api_key=CRIKK_API_KEY)
    
    # Use alternating voices for diversity
    voice_cycle = ['female_1', 'male_1', 'female_2', 'male_2']
    
    results = crikk.batch_synthesize(
        texts=crikk_texts,
        output_dir=CRIKK_OUTPUT_DIR,
        voice_ids=voice_cycle
    )
    
    # Save generation log
    results_df = pd.DataFrame(results)
    results_df.to_csv(AUGMENTATION_DATA_DIR / "crikk_generation_log.csv", index=False)
    
    success_count = sum(1 for r in results if r['success'])
    print(f"\n✅ Generated {success_count}/{len(results)} Crikk samples")
    
    return results_df

# Uncomment to run:
# crikk_results = generate_crikk_samples()

print("💡 Uncomment the line above to generate Crikk samples when API key is ready")


## 🤖 5. Orpheus Bangla TTS Generation

Orpheus is a **Transformer-based** TTS model (similar to LLaMA architecture for audio). Its artifact distribution is fundamentally different from VITS, making it ideal for diversifying our dataset.

Model: [asif00/orpheus-bangla-tts](https://huggingface.co/asif00/orpheus-bangla-tts)


In [8]:
import torch
import torchaudio
from transformers import AutoModelForCausalLM, AutoTokenizer

class OrpheusBanglaTTS:
    """
    Orpheus Bangla TTS - Transformer-based Bengali text-to-speech.
    
    This model uses a different architecture (Transformer) compared to VITS (CNN/Flow),
    providing diverse synthetic audio for training robust deepfake detectors.
    """
    
    def __init__(self, model_name: str = "asif00/orpheus-bangla-tts", device: str = None):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.tokenizer = None
        self.snac_model = None
        self.sample_rate = 24000  # Orpheus outputs at 24kHz
        
    def load_model(self):
        """Load the Orpheus model and SNAC decoder"""
        print(f"🔄 Loading Orpheus model on {self.device}...")
        
        # Load the Orpheus model
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.bfloat16 if self.device == "cuda" else torch.float32,
            device_map=self.device if self.device == "cuda" else None
        )
        
        if self.device == "cpu":
            self.model = self.model.to(self.device)
            
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        # Load SNAC model for audio decoding
        from snac import SNAC
        self.snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz")
        self.snac_model = self.snac_model.to(self.device)
        self.snac_model.eval()
        
        print("✅ Models loaded successfully!")
        
    def _process_tokens(self, tokens: list) -> torch.Tensor:
        """CORRECTED: Process tokens for Orpheus audio decoding"""
        
        # Orpheus might use a different approach - let's check the actual token ranges
        # Instead of filtering by offset, let's see what the model actually generates
        
        print(f"Raw generated tokens: {tokens}")
        
        # Remove obvious special tokens (start/end)
        special_tokens = [128000, 128009]  # BOS/EOS tokens
        audio_tokens = [t for t in tokens if t not in special_tokens]
        
        print(f"After removing special tokens: {audio_tokens}")
        print(f"Audio token range: {min(audio_tokens)} - {max(audio_tokens)}")
        
        # SNAC expects 7 codebooks. Let's check if we have enough tokens
        if len(audio_tokens) < 7:
            print(f"Not enough audio tokens: {len(audio_tokens)}")
            return torch.zeros((1, 7, 0), device=self.device, dtype=torch.long)
        
        # Try different approaches:
        
        # Approach 1: Take first 7 tokens as one time step
        if len(audio_tokens) >= 7:
            codes = torch.tensor(audio_tokens[:7]).unsqueeze(0).unsqueeze(0)  # [1, 1, 7]
            print(f"Using first 7 tokens: {codes.shape}")
            return codes.to(self.device)
        
        # If that doesn't work, the model might not be compatible with SNAC
        # In that case, we need to find Orpheus's native audio decoding method
        return torch.zeros((1, 7, 0), device=self.device, dtype=torch.long)
    
    def synthesize(self, text: str, output_path: str = None,
                   temperature: float = 0.6, max_new_tokens: int = 2048) -> Optional[np.ndarray]:
        """
        Synthesize Bengali text to speech.
        
        Args:
            text: Bengali text to synthesize
            output_path: Optional path to save the WAV file
            temperature: Sampling temperature (lower = more deterministic)
            max_new_tokens: Maximum tokens to generate
            
        Returns:
            Audio waveform as numpy array
        """
        if self.model is None:
            self.load_model()
        
        # Format prompt for Orpheus
        prompt = f"<|audio|>{text}<|eoa|>"
        
        # Tokenize
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        
        try:
            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            
            # Extract generated tokens
            generated_tokens = outputs[0][inputs['input_ids'].shape[1]:].tolist()
            
            # Process tokens for SNAC
            codes = self._process_tokens(generated_tokens)
            
            if codes.shape[2] == 0:
                print(f"⚠️ No audio tokens generated for: {text[:50]}...")
                return None
            
            # Decode with SNAC
            with torch.no_grad():
                audio = self.snac_model.decode(codes)
            
            audio_np = audio.squeeze().cpu().numpy()
            
            # Resample to 22050 Hz to match existing dataset
            audio_tensor = torch.from_numpy(audio_np).unsqueeze(0)
            resampler = torchaudio.transforms.Resample(self.sample_rate, 22050)
            audio_resampled = resampler(audio_tensor).squeeze().numpy()
            
            # Save if path provided
            if output_path:
                sf.write(output_path, audio_resampled, 22050)
            
            return audio_resampled
            
        except Exception as e:
            print(f"❌ Generation failed: {e}")
            return None

    # Debug version of the synthesize method
    def synthesize_debug(self, text: str, output_path: str = None,
                        temperature: float = 0.6, max_new_tokens: int = 2048) -> Optional[np.ndarray]:
        """
        Debug version that shows what tokens are generated.
        """
        if self.model is None:
            self.load_model()
        
        # Format prompt for Orpheus
        prompt = f"<|audio|>{text}<|eoa|>"
        
        # Tokenize
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        print(f"Input prompt: {prompt}")
        print(f"Input tokens: {inputs['input_ids'][0].tolist()}")
        
        try:
            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    # do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False  # Try deterministic generation first
                )
            
            # Extract generated tokens
            generated_tokens = outputs[0][inputs['input_ids'].shape[1]:].tolist()
            print(f"Generated tokens: {generated_tokens}")
            print(f"Number of generated tokens: {len(generated_tokens)}")
            
            # Show token ranges
            if generated_tokens:
                print(f"Min token: {min(generated_tokens)}, Max token: {max(generated_tokens)}")
                print(f"Tokens >= 128266: {[t for t in generated_tokens if t >= 128266]}")
            
            # Process tokens for SNAC
            processed = [t for t in generated_tokens if t not in [128000, 128009]]  # Remove special tokens
            if len(processed) % 7 != 0:
                processed = processed[:-(len(processed) % 7)]
            if processed:
                codes = torch.tensor(processed).reshape(-1, 7).T.unsqueeze(0).to(self.device)
            else:
                return None
            print(f"Processed codes shape: {codes.shape}")
            
            if codes.shape[2] == 0:
                print(f"⚠️ No audio tokens generated for: {text[:50]}...")
                return None
            
            # Decode with SNAC
            with torch.no_grad():
                audio = self.snac_model.decode(codes)
            
            audio_np = audio.squeeze().cpu().numpy()
            
            # Resample to 22050 Hz to match existing dataset
            audio_tensor = torch.from_numpy(audio_np).unsqueeze(0)
            resampler = torchaudio.transforms.Resample(self.sample_rate, 22050)
            audio_resampled = resampler(audio_tensor).squeeze().numpy()
            
            # Save if path provided
            if output_path:
                sf.write(output_path, audio_resampled, 22050)
            
            return audio_resampled
            
        except Exception as e:
            print(f"❌ Generation failed: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def batch_synthesize(self, texts: List[str], output_dir: Path,
                         start_idx: int = 0) -> List[Dict]:
        """
        Batch synthesize multiple texts.
        
        Args:
            texts: List of Bengali texts
            output_dir: Directory to save audio files
            start_idx: Starting index for filenames
            
        Returns:
            List of generation results with metadata
        """
        results = []
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        for i, text in enumerate(tqdm(texts, desc="Generating Orpheus samples")):
            filename = f"orpheus_{start_idx + i:05d}.wav"
            output_path = output_dir / filename
            
            audio = self.synthesize(text, str(output_path))
            
            results.append({
                'filename': filename,
                'text': text,
                'success': audio is not None,
                'path': str(output_path) if audio is not None else None
            })
            
            # Clear GPU memory periodically
            if i % 50 == 0 and self.device == "cuda":
                torch.cuda.empty_cache()
        
        return results

print("✅ OrpheusBanglaTTS class defined")
print(f"🖥️ Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


✅ OrpheusBanglaTTS class defined
🖥️ Device: CUDA


In [9]:
# ============================================
# ORPHEUS GENERATION - CONFIGURATION
# ============================================

ORPHEUS_TARGET_SAMPLES = 1500

# Sample transcripts for Orpheus generation (different from Crikk samples)
random.seed(123)  # Different seed for different samples
orpheus_transcripts = filtered_transcripts.sample(n=min(ORPHEUS_TARGET_SAMPLES, len(filtered_transcripts)))
orpheus_texts = orpheus_transcripts['text'].tolist()

print(f"📊 Prepared {len(orpheus_texts)} transcripts for Orpheus synthesis")
print(f"\n🖥️ GPU Available: {'✅ Yes' if torch.cuda.is_available() else '❌ No (will use CPU - slower)'}")


📊 Prepared 1500 transcripts for Orpheus synthesis

🖥️ GPU Available: ✅ Yes


In [14]:
%pip install hf_xet

  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl (2.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Test Orpheus with a single sample first
orpheus = OrpheusBanglaTTS()

test_text = "বাংলাদেশের রাজধানী ঢাকা"
print(f"🧪 Testing with: {test_text}")

try:
    orpheus.load_model()
    test_output = ORPHEUS_OUTPUT_DIR / "test_sample.wav"
    audio = orpheus.synthesize(test_text, str(test_output))
    
    if audio is not None:
        print(f"✅ Test successful! Audio saved to {test_output}")
        print(f"   Duration: {len(audio)/22050:.2f} seconds")
    else:
        print("⚠️ Test generation returned no audio")
except Exception as e:
    print(f"❌ Test failed: {e}")
    print("\nTroubleshooting:")
    print("1. Ensure you have enough GPU memory (8GB+ recommended)")
    print("2. Try reducing batch size or using CPU")
    print("3. Check internet connection for model download")


🧪 Testing with: বাংলাদেশের রাজধানী ঢাকা
🔄 Loading Orpheus model on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Models loaded successfully!
Raw generated tokens: [36278, 237, 11372, 243, 36278, 105, 28025, 233, 11372, 251, 11372, 97, 60008, 36278, 250, 87648, 53906, 107, 36278, 243, 73358, 60008, 36278, 237, 11372, 116, 60008, 36278, 117, 11372, 105, 60008, 36278, 101, 42412, 128009]
After removing special tokens: [36278, 237, 11372, 243, 36278, 105, 28025, 233, 11372, 251, 11372, 97, 60008, 36278, 250, 87648, 53906, 107, 36278, 243, 73358, 60008, 36278, 237, 11372, 116, 60008, 36278, 117, 11372, 105, 60008, 36278, 101, 42412]
Audio token range: 97 - 87648
Using first 7 tokens: torch.Size([1, 1, 7])
❌ Generation failed: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

⚠️ Test generation returned no audio


In [ ]:
# Generate Orpheus samples (batch generation)

def generate_orpheus_samples(num_samples: int = None):
    """Generate samples using Orpheus TTS"""
    orpheus = OrpheusBanglaTTS()
    
    texts_to_generate = orpheus_texts[:num_samples] if num_samples else orpheus_texts
    
    print(f"🎙️ Generating {len(texts_to_generate)} Orpheus samples...")
    print(f"   This may take a while. Estimated time: ~{len(texts_to_generate) * 5 / 60:.1f} minutes on GPU")
    
    results = orpheus.batch_synthesize(
        texts=texts_to_generate,
        output_dir=ORPHEUS_OUTPUT_DIR
    )
    
    # Save generation log
    results_df = pd.DataFrame(results)
    results_df.to_csv(AUGMENTATION_DATA_DIR / "orpheus_generation_log.csv", index=False)
    
    success_count = sum(1 for r in results if r['success'])
    print(f"\n✅ Generated {success_count}/{len(results)} Orpheus samples")
    
    return results_df

# Uncomment to run full generation:
# orpheus_results = generate_orpheus_samples()

# For testing with smaller batch:
# orpheus_results = generate_orpheus_samples(num_samples=10)

print("💡 Uncomment the generation line above to start Orpheus synthesis")


## 📊 6. Update Manifests with New Samples

After generating samples, we need to update the train/val/test manifests to include the new samples.

**Critical:** Keep some samples in the test set to measure generalization!


In [ ]:
def update_manifests(crikk_log_path: Path = None, orpheus_log_path: Path = None,
                     train_ratio: float = 0.8, val_ratio: float = 0.1):
    """
    Update manifests with newly generated samples.
    
    Args:
        crikk_log_path: Path to Crikk generation log CSV
        orpheus_log_path: Path to Orpheus generation log CSV
        train_ratio: Proportion for training set
        val_ratio: Proportion for validation set (rest goes to test)
    """
    test_ratio = 1.0 - train_ratio - val_ratio
    
    # Load existing manifests
    train_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "train_manifest.csv")
    val_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "val_manifest.csv")
    test_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "test_manifest.csv")
    
    new_entries = []
    
    # Process Crikk samples
    if crikk_log_path and crikk_log_path.exists():
        crikk_log = pd.read_csv(crikk_log_path)
        successful = crikk_log[crikk_log['success'] == True]
        
        for _, row in successful.iterrows():
            new_entries.append({
                'file_path': f"..\\Bangla Audio Deepfake Detection\\Data Augmentation\\Data\\crikk_deepfake\\{row['filename']}",
                'filename': row['filename'],
                'source': 'crikk',
                'label': 'fake',
                'label_id': 1
            })
        print(f"📦 Added {len(successful)} Crikk samples")
    
    # Process Orpheus samples
    if orpheus_log_path and orpheus_log_path.exists():
        orpheus_log = pd.read_csv(orpheus_log_path)
        successful = orpheus_log[orpheus_log['success'] == True]
        
        for _, row in successful.iterrows():
            new_entries.append({
                'file_path': f"..\\Bangla Audio Deepfake Detection\\Data Augmentation\\Data\\orpheus_deepfake\\{row['filename']}",
                'filename': row['filename'],
                'source': 'orpheus',
                'label': 'fake',
                'label_id': 1
            })
        print(f"📦 Added {len(successful)} Orpheus samples")
    
    if not new_entries:
        print("⚠️ No new samples to add. Generate samples first!")
        return
    
    # Convert to DataFrame and shuffle
    new_df = pd.DataFrame(new_entries)
    new_df = new_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Split into train/val/test
    n_total = len(new_df)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    
    new_train = new_df.iloc[:n_train]
    new_val = new_df.iloc[n_train:n_train + n_val]
    new_test = new_df.iloc[n_train + n_val:]
    
    print(f"\n📊 Split Distribution:")
    print(f"   Train: {len(new_train)} ({train_ratio*100:.0f}%)")
    print(f"   Val:   {len(new_val)} ({val_ratio*100:.0f}%)")
    print(f"   Test:  {len(new_test)} ({test_ratio*100:.0f}%)")
    
    # Append to existing manifests
    train_updated = pd.concat([train_df, new_train], ignore_index=True)
    val_updated = pd.concat([val_df, new_val], ignore_index=True)
    test_updated = pd.concat([test_df, new_test], ignore_index=True)
    
    # Save updated manifests
    train_updated.to_csv(AUGMENTATION_MANIFEST_DIR / "train_manifest.csv", index=False)
    val_updated.to_csv(AUGMENTATION_MANIFEST_DIR / "val_manifest.csv", index=False)
    test_updated.to_csv(AUGMENTATION_MANIFEST_DIR / "test_manifest.csv", index=False)
    
    # Also update the full dataset manifest
    dataset_df = pd.concat([train_updated, val_updated, test_updated], ignore_index=True)
    dataset_df.to_csv(AUGMENTATION_MANIFEST_DIR / "dataset_manifest.csv", index=False)
    
    print("\n✅ Manifests updated successfully!")
    print("\n📊 Updated Dataset Statistics:")
    print("="*50)
    print(f"Training set: {len(train_updated)} samples")
    print(f"Validation set: {len(val_updated)} samples")
    print(f"Test set: {len(test_updated)} samples")
    
    # Show source distribution
    print("\n📈 Source Distribution in Updated Training Set:")
    print(train_updated['source'].value_counts())
    
    return train_updated, val_updated, test_updated

print("✅ update_manifests function defined")


In [ ]:
# Update manifests with generated samples
# Run this after generating Crikk and/or Orpheus samples

crikk_log = AUGMENTATION_DATA_DIR / "crikk_generation_log.csv"
orpheus_log = AUGMENTATION_DATA_DIR / "orpheus_generation_log.csv"

print("📋 Checking for generation logs...")
print(f"   Crikk log: {'✅ Found' if crikk_log.exists() else '❌ Not found'}")
print(f"   Orpheus log: {'✅ Found' if orpheus_log.exists() else '❌ Not found'}")

if crikk_log.exists() or orpheus_log.exists():
    updated_manifests = update_manifests(
        crikk_log_path=crikk_log if crikk_log.exists() else None,
        orpheus_log_path=orpheus_log if orpheus_log.exists() else None,
        train_ratio=0.8,
        val_ratio=0.1
    )
else:
    print("\n⚠️ No generation logs found. Please run the generation cells first.")


## 📈 7. Verification & Summary


In [ ]:
def verify_augmentation():
    """Verify the augmentation results and display summary statistics"""
    
    print("🔍 Verification Report")
    print("="*60)
    
    # Check generated files
    crikk_files = list(CRIKK_OUTPUT_DIR.glob("*.wav"))
    orpheus_files = list(ORPHEUS_OUTPUT_DIR.glob("*.wav"))
    
    print(f"\n📁 Generated Audio Files:")
    print(f"   Crikk:   {len(crikk_files)} files")
    print(f"   Orpheus: {len(orpheus_files)} files")
    print(f"   Total:   {len(crikk_files) + len(orpheus_files)} files")
    
    # Load manifests
    if (AUGMENTATION_MANIFEST_DIR / "train_manifest.csv").exists():
        train_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "train_manifest.csv")
        val_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "val_manifest.csv")
        test_df = pd.read_csv(AUGMENTATION_MANIFEST_DIR / "test_manifest.csv")
        
        print(f"\n📊 Manifest Statistics:")
        print(f"   Training:   {len(train_df)} samples")
        print(f"   Validation: {len(val_df)} samples")
        print(f"   Test:       {len(test_df)} samples")
        
        # Source distribution
        all_df = pd.concat([train_df, val_df, test_df])
        print(f"\n📈 Source Distribution (All Splits):")
        source_counts = all_df['source'].value_counts()
        for source, count in source_counts.items():
            pct = count / len(all_df) * 100
            print(f"   {source}: {count} ({pct:.1f}%)")
        
        # Label distribution
        print(f"\n🏷️ Label Distribution:")
        for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
            real_count = len(split_df[split_df['label'] == 'real'])
            fake_count = len(split_df[split_df['label'] == 'fake'])
            print(f"   {split_name}: Real={real_count}, Fake={fake_count}")
        
        # New sources in test set (critical for generalization testing)
        print(f"\n🧪 Test Set Composition (Critical for Generalization):")
        test_sources = test_df['source'].value_counts()
        for source, count in test_sources.items():
            if source in ['crikk', 'orpheus']:
                print(f"   ⭐ {source}: {count} (NEW - unseen algorithm)")
            else:
                print(f"   {source}: {count}")
    else:
        print("\n⚠️ Manifests not updated yet. Run the generation and update cells.")
    
    print("\n" + "="*60)
    print("✅ Verification complete!")

verify_augmentation()


In [ ]:
# Sample audio playback (if running in Jupyter with audio support)
from IPython.display import Audio, display

def play_samples():
    """Play sample audio from each source"""
    samples = {
        'Crikk': list(CRIKK_OUTPUT_DIR.glob("*.wav")),
        'Orpheus': list(ORPHEUS_OUTPUT_DIR.glob("*.wav"))
    }
    
    for source, files in samples.items():
        if files:
            sample_file = files[0]
            print(f"\n🎵 {source} Sample: {sample_file.name}")
            try:
                display(Audio(str(sample_file)))
            except Exception as e:
                print(f"   (Audio playback not available: {e})")
        else:
            print(f"\n⚠️ No {source} samples found")

# Uncomment to play samples:
# play_samples()


## 🚀 8. Next Steps

After completing this augmentation:

1. **Verify Audio Quality**: Listen to random samples from each source to ensure quality
2. **Check Manifest Integrity**: Ensure all file paths are valid
3. **Proceed to Baseline Model**: Build RawNet2 using the updated manifests from `Data Augmentation/Manifest/`

### Key Points:
- ✅ New fake samples from **Crikk** (proprietary/black-box)
- ✅ New fake samples from **Orpheus** (Transformer-based)
- ✅ Test set includes samples from NEW sources (generalization testing)
- ✅ Manifests updated and ready for training

---

**Important Paths for Training:**
- Training Manifest: `Data Augmentation/Manifest/train_manifest.csv`
- Validation Manifest: `Data Augmentation/Manifest/val_manifest.csv`
- Test Manifest: `Data Augmentation/Manifest/test_manifest.csv`


In [ ]:
# Final summary
print("\n" + "🎉"*20)
print("\n   BENGALI-WILD AUGMENTATION COMPLETE!")
print("\n" + "🎉"*20)

print("\n📍 Output Locations:")
print(f"   Crikk Audio:    {CRIKK_OUTPUT_DIR}")
print(f"   Orpheus Audio:  {ORPHEUS_OUTPUT_DIR}")
print(f"   Updated Manifests: {AUGMENTATION_MANIFEST_DIR}")

print("\n🔜 Next Step: Build Baseline Model (RawNet2)")
